# 🏰 Obrázok → 3D model (Hunyuan3D-2)

Z obrázka budovy (napr. z Forge) spraví 3D model **s farbami** ako `.glb` súbor pre Godot.

**Postup:**
1. Skontroluj, že vpravo dole je napísané **T4**. Ak nie: v hornom menu **Runtime → Zmeniť typ runtime → T4 GPU → Uložiť**.
2. Klikni **▶ Spustiť všetko** (tlačidlo hore pod menu, alebo **Ctrl+F9**). Ak vyskočí varovanie, že notebook nie je od Googlu, daj **Aj tak spustiť**.
3. Bunka 1 inštaluje cca 5–10 minút. Potom sa v bunke 2 objaví tlačidlo **Vybrať súbory**.
4. Vyber obrázok (alebo viac naraz). Pozadie sa odstráni samo.
5. Počkaj (prvýkrát sa sťahujú modely, potom cca 3–6 minút na obrázok). `.glb` sa stiahne sám (prehliadač sa môže spýtať, či povoliť viac súborov → povoliť).
6. Ďalší obrázok: klikni na ▶ pri bunke **2**.

**Tipy:** Najlepšie funguje jedna budova v strede, celá v zábere, na jednoduchom pozadí, mierne zhora zboku.
Ak Colab napíše, že došla GPU, skús neskôr (bezplatný limit je pár hodín denne).

In [ ]:
#@title 1. Inštalácia (raz, cca 5–10 min)
import os, subprocess, sys
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("❌ Nie je zapnutá GPU! Runtime → Zmeniť typ runtime → T4 GPU → Uložiť, potom znova Spustiť všetko.")
print("✅ GPU:", gpu)

def sh(cmd, povinne=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 and povinne:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit("❌ Chyba pri: " + cmd)

ZNACKA = "/content/.instalacia_ok2"
if os.path.exists(ZNACKA):
    print("✅ Inštalácia už je hotová")
else:
    os.chdir("/content")
    if not os.path.exists("/content/Hunyuan3D-2"):
        sh("git clone --depth 1 https://github.com/Tencent/Hunyuan3D-2")
    os.chdir("/content/Hunyuan3D-2")
    print("⏳ Inštalujem knižnice...")
    # Už nainštalované knižnice (numpy, Pillow, torch...) nesmú meniť verziu. Meniť sa smú len tieto:
    menit = {"diffusers", "transformers", "huggingface-hub", "huggingface_hub", "tokenizers", "safetensors", "accelerate", "hf-xet", "hf_xet"}
    zoznam = subprocess.run([sys.executable, "-m", "pip", "list", "--format=freeze"], capture_output=True, text=True).stdout
    riadky = [r for r in zoznam.splitlines() if "==" in r and r.split("==")[0].lower() not in menit]
    open("/content/obmedzenia.txt", "w").write("\n".join(riadky) + "\n")
    sh("pip install -q -c /content/obmedzenia.txt diffusers==0.32.2 transformers==4.49.0 'huggingface_hub<0.31' "
       "accelerate einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime ninja pybind11")
    print("⏳ Kompilujem renderer textúr (najdlhší krok)...")
    os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
    sh("pip install -q --no-build-isolation ./hy3dgen/texgen/custom_rasterizer")
    sh("cd hy3dgen/texgen/differentiable_renderer && python setup.py build_ext --inplace", povinne=False)
    open(ZNACKA, "w").write("ok")
    print("✅ Inštalácia hotová")

# Samotné generovanie beží ako samostatný program (nezhodí Colab, keď dôjde pamäť).
GEN = r"""
import os, sys, gc, time, glob
os.environ["USE_TF"] = "0"; os.environ["TRANSFORMERS_NO_TF"] = "1"
sys.path.insert(0, "/content/Hunyuan3D-2"); os.chdir("/content/Hunyuan3D-2")
import torch, trimesh
from PIL import Image
vstup, vystup, max_tv, textura, kroky, rozl = sys.argv[1], sys.argv[2], int(sys.argv[3]), sys.argv[4] == "1", int(sys.argv[5]), int(sys.argv[6])
obrazky = sorted(glob.glob(vstup + "/*"))
t0 = time.time()

print("⏳ Odstraňujem pozadie...", flush=True)
from hy3dgen.rembg import BackgroundRemover
rembg = BackgroundRemover()
pripravene = []
for p in obrazky:
    img = Image.open(p)
    if img.mode == "RGBA" and img.getextrema()[3][0] < 255:
        img = img.convert("RGBA")
    else:
        img = rembg(img.convert("RGB"))
    pripravene.append((os.path.splitext(os.path.basename(p))[0], img))
del rembg; gc.collect()

print("⏳ Načítavam model tvaru...", flush=True)
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline, FaceReducer, FloaterRemover, DegenerateFaceRemover
shape = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained("tencent/Hunyuan3D-2", subfolder="hunyuan3d-dit-v2-0")
tvary = []
for meno, img in pripravene:
    mesh = shape(image=img, num_inference_steps=kroky, octree_resolution=rozl, num_chunks=8000,
                 generator=torch.manual_seed(1234))[0]
    mesh = FloaterRemover()(mesh); mesh = DegenerateFaceRemover()(mesh)
    mesh = FaceReducer()(mesh, max_facenum=max_tv)
    tvary.append((meno, img, mesh))
    if not textura:
        mesh.export(f"{vystup}/{meno}.glb")
    print(f"   tvar {meno} hotový ({time.time()-t0:.0f} s)", flush=True)
del shape; gc.collect(); torch.cuda.empty_cache()

if textura:
    print("⏳ Načítavam model textúr...", flush=True)
    from hy3dgen.texgen import Hunyuan3DPaintPipeline
    paint = Hunyuan3DPaintPipeline.from_pretrained("tencent/Hunyuan3D-2")
    for meno, img, mesh in tvary:
        try:
            paint(mesh, image=img).export(f"{vystup}/{meno}.glb")
            print(f"   textúra {meno} hotová ({time.time()-t0:.0f} s)", flush=True)
        except Exception as e:
            print(f"   ⚠️ textúra {meno} zlyhala ({e}), ukladám bez farieb", flush=True)
            mesh.export(f"{vystup}/{meno}_bez_farieb.glb")
        gc.collect(); torch.cuda.empty_cache()
print("HOTOVO", flush=True)
"""
open("/content/generuj.py", "w").write(GEN)

In [ ]:
#@title 2. Nahraj obrázok → stiahni GLB (pre ďalšie obrázky spúšťaj znova len túto bunku)
max_trojuholnikov = 30000  #@param {type:"integer"}
s_texturou = True  #@param {type:"boolean"}
kvalita = "normálna"  #@param ["rýchla", "normálna", "vysoká"]

import os, shutil, subprocess, sys, glob
from google.colab import files
kroky, rozlisenie = {"rýchla": (25, 256), "normálna": (40, 320), "vysoká": (50, 384)}[kvalita]
for d in ("/content/vstup", "/content/vystup"):
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
nahrane = files.upload()
for meno, data in nahrane.items():
    open(os.path.join("/content/vstup", meno), "wb").write(data)
if nahrane:
    proc = subprocess.Popen([sys.executable, "/content/generuj.py", "/content/vstup", "/content/vystup",
                             str(max_trojuholnikov), "1" if s_texturou else "0", str(kroky), str(rozlisenie)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    posledne = []
    for riadok in proc.stdout:
        posledne = (posledne + [riadok])[-30:]
        if riadok.startswith(("⏳", "   ", "HOTOVO")):
            print(riadok, end="")
    proc.wait()
    vysledky = sorted(glob.glob("/content/vystup/*.glb"))
    for cesta in vysledky:
        files.download(cesta)
    if proc.returncode == 0:
        print(f"✅ Hotovo, sťahujem {len(vysledky)} súbor(y)")
    elif proc.returncode in (-9, 137):
        print("❌ Colabu došla pamäť. Skús kvalitu 'rýchla' alebo menej obrázkov naraz.")
    else:
        print("❌ Chyba. Pošli screenshot tohto textu:\n" + "".join(posledne))